In [1]:
import dgl
from dgl import DGLGraph
from dgl.nn.pytorch.conv import GATConv

from src.backends.cusparse_backend.conv import _СuSparseMatMulConv
from src.backends.fusegnn_backend.conv import garGATConv, garGCNConv, gasGCNConv, gasGATConv

# cusparse_conv = _СuSparseMatMulConv("both", 0)

Detected CUDA files, patching ldflags
Emitting ninja build file /home/daniilkk/repos/Accelerating-GNNs/build/cusparse/build.ninja...
/home/daniilkk/repos/Accelerating-GNNs/.venv/lib/python3.11/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module cuda_kernels...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


ninja: no work to do.


Loading extension module cuda_kernels...
Detected CUDA files, patching ldflags
Emitting ninja build file /home/daniilkk/repos/Accelerating-GNNs/build/fuseGNN/build.ninja...
/home/daniilkk/repos/Accelerating-GNNs/.venv/lib/python3.11/site-packages/torch/utils/cpp_extension.py:1965: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fgnn_ops...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
Loading extension module fgnn_ops...


ninja: no work to do.


In [2]:
from typing import Optional, Tuple

import torch


def coo_to_csr(row_indices, col_indices, values=None, shape=None, device="cuda"):
    """
    Convert COO format to CSR format.

    Args:
        row_indices: Row indices of non-zero elements
        col_indices: Column indices of non-zero elements
        values: Values (None = all ones for unweighted graphs)
        shape: (n_rows, n_cols) or None to infer
        device: 'cuda' or 'cpu'

    Returns:
        (row_ptr, col_indices, values) all on specified device
    """
    row_indices = torch.tensor(row_indices, dtype=torch.int64, device=device)
    col_indices = torch.tensor(col_indices, dtype=torch.int64, device=device)
    values = torch.ones(len(row_indices), device=device) if values is None else torch.tensor(values, device=device)

    # Infer shape if needed
    if shape is None:
        n_rows = int(row_indices.max().item()) + 1 if len(row_indices) > 0 else 0
    else:
        n_rows = shape[0]

    # Handle empty
    if len(row_indices) == 0:
        return torch.zeros(n_rows + 1, dtype=torch.int64, device=device), col_indices, values

    # Sort by row then column
    sort_idx = torch.argsort(row_indices * (col_indices.max() + 1) + col_indices)
    row_indices = row_indices[sort_idx]
    col_indices = col_indices[sort_idx]
    values = values[sort_idx]

    # Build row_ptr
    row_ptr = torch.zeros(n_rows + 1, dtype=torch.int64, device=device)
    row_ptr.scatter_add_(0, row_indices + 1, torch.ones_like(row_indices))
    torch.cumsum(row_ptr, dim=0, out=row_ptr)

    return row_ptr.int(), col_indices.int(), values

In [3]:
import numpy as np
import torch


def create_simple_test_data():
    """
    Creates a simple graph with 5 nodes
    """
    num_nodes = 6
    feature_dim = 7

    # Node features (random for simplicity)
    # x = torch.randn(num_nodes, feature_dim)
    x = torch.eye(num_nodes)

    #           v
    #   > 0 <-- 1 --> 5
    #     ^     ^
    #     |     |
    #     v     v
    #   > 2 <-> 3 <-> 4
    #     ^     ^     ^

    # Edge list in COO format
    edges = [
        [0, 1],
        [0, 2],
        [2, 0],
        [1, 3],
        [3, 1],
        [2, 3],
        [3, 2],
        [3, 4],
        [4, 3],
        [5, 1],
        [0, 0],
        [1, 1],
        [2, 2],
        [3, 3],
        [4, 4],
    ]

    # edges = [(e2, e1) for e1, e2 in edges]
    edge_index = torch.tensor(edges, dtype=torch.long).t()  # Shape: [2, num_edges]

    # Optional: edge weights (all 1.0 for simplicity)
    edge_weight = torch.randint(1, 10, (edge_index.size(1),)).to(torch.float32)
    # edge_weight = torch.ones(edge_index.size(1), dtype=torch.float32) * 3

    return x.cuda(), edge_index.cuda(), edge_weight.cuda(), num_nodes


def create_simple_test_data2():
    """
    Creates a simple graph with 5 nodes
    """
    num_nodes = 3
    feature_dim = 7

    # Node features (random for simplicity)
    # x = torch.randn(num_nodes, feature_dim)
    x = torch.eye(num_nodes)

    #   0 <-- 1 <-> 2

    # Edge list in COO format
    edges = [
        [0, 1],
        [1, 2],
        [2, 1],
        # [1, 0],
    ]

    edge_index = torch.tensor(edges, dtype=torch.long).t()  # Shape: [2, num_edges]

    # Optional: edge weights (all 1.0 for simplicity)
    edge_weight = torch.randint(1, 2, (edge_index.size(1),)).to(torch.float32)
    # edge_weight = torch.ones(edge_index.size(1), dtype=torch.float32) * 3

    return x.cuda(), edge_index.cuda(), edge_weight.cuda(), num_nodes


x, edge_index, edge_weight, num_nodes = create_simple_test_data2()

print(f"{x=}")
# print()
# print(f"{edge_weight=}")
# print(f'{edge_index=}')

g_dgl = dgl.graph((edge_index[1], edge_index[0]), num_nodes=num_nodes)
# print(g_dgl.device)
# g.ndata['feat'] = x

# print()

x=tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]], device='cuda:0')


In [6]:
fgnn_conv = gasGATConv(in_channels=x.shape[1], cached=False, flow="target_to_source").to('cuda')
dgl_conv = GATConv(in_feats=x.shape[1], out_feats=x.shape[1], num_heads=1).to('cuda')

In [7]:
fgnn_output = fgnn_conv(x, edge_index)

graph_csr = coo_to_csr(edge_index[0], edge_index[1], edge_weight, shape=(num_nodes, num_nodes))
dgl_output = dgl_conv(g_dgl, x)

print("fgnn")
print(fgnn_output)
print()
print("dgl")
print(dgl_output)
print()
print("MATCH" if torch.allclose(fgnn_output, dgl_output) else "MISMATCH")
print()

fgnn
tensor([[-0.0128, -0.2626, -0.3740],
        [ 0.1380,  0.5617, -0.3222],
        [-0.0105, -0.2154, -0.3068]], device='cuda:0',
       grad_fn=<fusedGASAggV2Backward>)

dgl
tensor([[[-0.6862,  0.8037, -0.5013]],

        [[-0.5420, -1.1343, -1.4086]],

        [[-0.6862,  0.8037, -0.5013]]], device='cuda:0',
       grad_fn=<AddBackward0>)

MISMATCH



/var/tmp/ipykernel_346973/2048785432.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  row_indices = torch.tensor(row_indices, dtype=torch.int64, device=device)
/var/tmp/ipykernel_346973/2048785432.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  col_indices = torch.tensor(col_indices, dtype=torch.int64, device=device)
/var/tmp/ipykernel_346973/2048785432.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  values = torch.ones(len(row_indices), device=device) if values is None else torch.tensor(values, device=device)


In [10]:
import torch

from src.backends.fusegnn_backend.functional import gcn_gas_edge_weight
from src.backends.fusegnn_backend.utils import fgnn_gcn

# Simple test
src_index = torch.tensor([1, 2, 1], dtype=torch.int32, device="cuda")
tar_index = torch.tensor([0, 1, 2], dtype=torch.int32, device="cuda")
num_nodes = 3
edge_weight = torch.ones(3, dtype=torch.float32, device="cuda")

print("Input:")
print(f"src_index: {src_index}")
print(f"tar_index: {tar_index}")
print(f"edge_weight: {edge_weight}")
print()

edge_weight_out, degree = gcn_gas_edge_weight(src_index, tar_index, num_nodes, edge_weight, True)

print("Output:")
print(f"edge_weight_out: {edge_weight_out}")
print(f"degree (tar_degree): {degree}")
print(f"self_edge_weight (1/sqrt(degree)): {1.0 / torch.sqrt(degree)}")

Input:
src_index: tensor([1, 2, 1], device='cuda:0', dtype=torch.int32)
tar_index: tensor([0, 1, 2], device='cuda:0', dtype=torch.int32)
edge_weight: tensor([1., 1., 1.], device='cuda:0')

Output:
edge_weight_out: tensor([0.7071, 1.0000, 0.7071], device='cuda:0')
degree (tar_degree): tensor([1., 1., 1.], device='cuda:0')
self_edge_weight (1/sqrt(degree)): tensor([1., 1., 1.], device='cuda:0')
